# 4. Data Cleaning

#### Objective

This notebook cleans the raw dataset, fixing data types, handling missing
values, and validating data integrity, to produce a reliable dataset for the
Exploratory Data Analysis phase.

## 4.1 Load Data

#### Objective

This notebook is self-contained and reloads the raw dataset independently,
so it can be run without depending on the state of `01_data_understanding.ipynb`.

In [1]:
import pandas as pd
import numpy as np

data_path = "../data/raw/"

df = pd.read_csv(data_path + "WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.shape

(7043, 21)

In [2]:
cleaning_log = pd.DataFrame(columns=[
    "step", "table", "column", "issue", "action", "reason", "validation"
])

## 4.2 Fix Data Types

#### Objective

`SeniorCitizen` is stored as integer (0/1) while all other binary attributes
in the dataset (`Partner`, `Dependents`, etc.) are stored as "Yes"/"No"
strings. This section standardizes `SeniorCitizen` to the same "Yes"/"No"
format for consistency across the dataset, which simplifies grouping and
visualization in later phases.

In [3]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

df["SeniorCitizen"].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

#### Observation

`SeniorCitizen` now uses the same "Yes"/"No" format as the other binary
categorical columns, making it consistent for groupby operations and
visualizations throughout the EDA phase.

## 4.3 Convert TotalCharges to Numeric

#### Objective

`TotalCharges` was read as an `object` (text) column during Data
Understanding. This section investigates why, converts it to numeric, and
handles any resulting missing values.

In [4]:
non_numeric = df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()]

non_numeric[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


#### Analysis

The non-numeric values are expected to be blank strings (`" "`), all
belonging to customers with `tenure = 0`. This makes sense: these are brand
new customers who have not yet been billed, so they have no total charges
accumulated yet. This is not a data quality error — it reflects a valid
business state — but the column still needs to be numeric for analysis.

In [5]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"].isna().sum()

np.int64(11)

In [6]:
df.loc[df["tenure"] == 0, "TotalCharges"] = df.loc[df["tenure"] == 0, "TotalCharges"].fillna(0)

df["TotalCharges"].isna().sum()

np.int64(0)

#### Decision

Missing `TotalCharges` values, all corresponding to `tenure = 0`, are filled
with `0` rather than dropped or imputed with a statistical estimate. This is
the logically correct value for a customer who has not yet been billed, and
preserves these records for the rest of the analysis.

In [7]:
cleaning_log.loc[len(cleaning_log)] = [
    "4.3", "df", "TotalCharges",
    "Blank strings for tenure=0 customers (not billed yet)",
    "Converted to numeric, filled with 0 for tenure=0 rows",
    "Blank value reflects a valid business state, not missing data",
    "0 missing values remain"
]

cleaning_log

,step,table,column,issue,action,reason,validation
0,4.3,df,TotalCharges,Blank strings for tenure=0 customers (not bill...,"Converted to numeric, filled with 0 for tenure...","Blank value reflects a valid business state, n...",0 missing values remain


## 4.4 Missing Value Treatment

#### Objective

A full missing value check across all columns, now that `TotalCharges` has
been converted and treated.

In [8]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

#### Observation

No missing values are expected to remain in any column after the
`TotalCharges` treatment in the previous section.

## 4.5 Duplicate Handling

#### Objective

Checking for fully duplicated rows and duplicated `customerID` values, since
each row should represent one unique customer.

In [9]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customerID:", df["customerID"].duplicated().sum())

Fully duplicated rows: 0
Duplicated customerID: 0


#### Observation

No duplicate customers are expected in this dataset, based on the initial
check performed during Data Understanding.

## 4.6 Check for Illogical Values

#### Objective

This section checks whether numeric relationships in the data are internally
consistent — specifically, whether `TotalCharges` is roughly consistent with
`tenure × MonthlyCharges`, and whether any charges or tenure values are
negative.

In [10]:
print("Negative tenure:", (df["tenure"] < 0).sum())
print("Negative MonthlyCharges:", (df["MonthlyCharges"] < 0).sum())
print("Negative TotalCharges:", (df["TotalCharges"] < 0).sum())

Negative tenure: 0
Negative MonthlyCharges: 0
Negative TotalCharges: 0


In [11]:
df["expected_total"] = df["tenure"] * df["MonthlyCharges"]
df["total_charges_diff"] = (df["TotalCharges"] - df["expected_total"]).abs()

df[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "expected_total", "total_charges_diff"]].sort_values(
    by="total_charges_diff", ascending=False
).head(10)

,customerID,tenure,MonthlyCharges,TotalCharges,expected_total,total_charges_diff
1418,9350-VLHMB,67,89.55,6373.10,5999.85,373.25
1746,3963-RYFNS,72,116.45,8013.55,8384.40,370.85
6596,0083-PIVIK,64,81.25,5567.55,5200.00,367.55
1274,7182-OVLBJ,62,101.15,6638.35,6271.30,367.05
1997,0266-CLZKZ,67,105.65,6717.90,7078.55,360.65
152,1679-JRFBR,70,108.15,7930.55,7570.50,360.05
2337,4612-THJBS,56,104.75,5510.65,5866.00,355.35
3200,0895-DQHEW,54,104.30,5278.15,5632.20,354.05
3634,3258-SYSWS,72,113.80,7845.80,8193.60,347.80
2264,7176-WRTNX,70,114.95,7711.25,8046.50,335.25


#### Analysis

Some difference between `TotalCharges` and `tenure × MonthlyCharges` is
expected, since `MonthlyCharges` reflects the customer's *current* rate,
while historical charges may have varied over time (e.g. due to plan changes
or promotions). Large discrepancies are treated as a normal characteristic of
the data rather than an error, since there is no evidence of billing history
per month to validate against. No rows are removed based on this check.

In [12]:
df = df.drop(columns=["expected_total", "total_charges_diff"])

## 4.7 Target Variable Check

#### Objective

Confirming that the `Churn` target column contains only the two expected
values, with no missing or unexpected entries.

In [13]:
df["Churn"].value_counts(dropna=False)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

#### Observation

The target variable is expected to contain exactly two values, "Yes" and
"No", with no missing entries, confirming it is ready to be used as-is for
both the Statistical Analysis and, later, model training in Phase 2.

## 4.8 Data Leakage Check

#### Objective

Data leakage occurs when a feature contains information that would not be
available at prediction time, or that trivially encodes the target. This
section reviews the dataset's columns for such risks before they are carried
into Feature Engineering and, later, modeling.

In [14]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

#### Analysis

- `customerID` is a unique identifier with no predictive meaning and should
  be excluded from any modeling features (kept only for record-keeping).
- No column directly encodes `Churn` (e.g. no "cancellation date" or
  "reason for leaving" field exists in this dataset).
- `tenure` deserves a conceptual caveat rather than a technical leak: for
  churned customers, `tenure` represents the time until they churned, while
  for active customers it represents time as of the snapshot date. This is a
  valid and commonly used feature in churn modeling, but it means `tenure`
  is not "prior to churn" in the same sense for every row — this should be
  kept in mind when interpreting its importance in Phase 2.

No columns are removed at this stage; `customerID` will be excluded at the
Preprocessing step in Phase 2, not from the dataset itself.

## 4.9 Save Cleaned Dataset

#### Objective

Saving the cleaned dataset to `data/processed/` allows the remaining
notebooks (`03_eda.ipynb` onward) to load it directly without repeating the
cleaning steps.

In [19]:
df.to_csv(processed_path + "telco_cleaned.csv", index=False)

df.shape

(7043, 21)

#### Summary

The dataset was cleaned by standardizing `SeniorCitizen` to a consistent
Yes/No format, converting `TotalCharges` to numeric and filling missing
values for new customers with 0, and validating the absence of duplicates,
negative values, and target-encoding data leakage. The cleaned dataset is
saved to `data/processed/telco_cleaned.csv` and is ready for Exploratory
Data Analysis in `03_eda.ipynb`.